# Simulation study for Bayesian logistic regression example

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))  
sys.path.append(project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
import pickle
from algorithms import sig, log_p_laplace, soul_mh, soul_mh_decay, proximal_map_laplace_iterative, proximal_map_laplace_approx, mypipla, mypgd
#pip_ula, prox_pgd, proximal_map_laplace_iteration_total, pipgla, proximal_map_laplace_approx_total
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

os.chdir(project_root)

# Obtain synthetic dataset for Laplace prior case

In [ ]:
from scipy.stats import laplace, bernoulli
# Data and design matrix
np.random.seed(3)
design_matrix = np.random.uniform(low = -1.0, high = 1.0, size = (900, 50))
x_unknown = laplace.rvs(loc = -4, size = (50, 1))
parameter_bernoulli = sig(np.matmul(design_matrix, x_unknown))
data_experiment = bernoulli.rvs(np.array(parameter_bernoulli[:, 0]), size = 900)
labels = np.expand_dims(data_experiment, axis=1)
theta_true = -4

# SOUL Metropolis-Hastings algorithm 

In [ ]:
# One run test
# --- SOUL Optimization Setup ---
D = 50       # Dimensionality of latent variables (x)
T = 800      # Outer optimization steps
M = 25       # MH steps per outer loop
B = 5       # Burn-in steps
delta_step = 0.08
proposal_std = 0.05
b_scale = 1.0 # Scale parameter for the Laplace prior

# Initializations
th0 = np.array([[-15.0]]) 
x0_M = np.zeros((D, 1))   # Initial latent variables vector

# Run the adapted algorithm
th_list, x_samples = soul_mh(
    log_p=log_p_laplace,
    th0=th0,
    x0_M=x0_M,
    y_l=design_matrix,
    y_f=labels,
    T=T,
    M=M,
    B=B,
    D=D,
    delta_step=delta_step,
    proposal_std=proposal_std,
    b=b_scale
)

print("Final estimated theta:", th_list[-1][0, 0])
print("True theta (loc parameter): -4.0")

In [ ]:
# --- Experiment Parameters ---
T = 500  # Outer steps 
B = 5               
M = 20 

# Tuning hyper-parameters for SOUL-MH
delta_step = 0.08    # Theta learning rate
proposal_std = 0.05  # MH proposal standard deviation
b_scale = 1.0        # Laplace prior scale

fig = plt.figure(figsize=(10, 6))
theta_soul = []
X_soul = []

for run_idx in range(7):
    # Randomly initialize theta0 between -15 and 10
    theta0_val = np.random.randint(-15, 10)
    th0 = np.array([[float(theta0_val)]])
    
    # Initialize X0 drawn from Gaussian centered at theta0, shape (D, M)
    # where D is feature dimension (50 from Paula's dataset)
    D = design_matrix.shape[1]
    X0_M = np.random.normal(loc=theta0_val, scale=1.0, size=(D, M))
    
    th_list, x_values = soul_mh(
        log_p=log_p_laplace,
        th0=th0,
        x0_M=X0_M,
        y_l=design_matrix,
        y_f=labels,
        T=T,
        M=M,
        B=B,
        D=D,
        delta_step=delta_step,
        proposal_std=proposal_std,
        b=b_scale
    )
    
    # Extract flattened array of theta trajectory across iterations
    th_trajectory = np.array([t[0, 0] for t in th_list])
    
    theta_soul.append(th_trajectory)
    X_soul.append(x_values)
    
    # Plot trajectory for this run
    plt.plot(th_trajectory, label=f'Run {run_idx + 1} (Init $\\theta_0={theta0_val}$)')

# Reference line for ground truth mean
plt.axhline(y=np.mean(x_unknown), color='black', linestyle='dashed', label='True Mean $\\theta$')

plt.title('SOUL-MH Convergence Across Multiple Initializations')
plt.xlabel('Iteration (T)')
plt.ylabel('Theta Estimate ($\\theta$)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.show()
fig.savefig('soul_mh_convergence.pdf', format='pdf')

Because this loop was quite long to run, and all runs are independent, I tried to parallelise the runs:

In [ ]:
pip install joblib

In [ ]:
from joblib import Parallel, delayed

# tried parallelisation to go faster ! success :))

# --- Experiment Parameters ---
T = 800  # Outer steps 
B = 5               
M = 35

# Tuning hyper-parameters for SOUL-MH
delta_step = 0.05    # Theta learning rate
proposal_std = 0.08  # MH proposal standard deviation
b_scale = 1.0        # Laplace prior scale
#gamma_val = 0.9995

def run_single_soul_mh_decay(run_idx):
    theta0_val = np.random.randint(-15, 10)
    th0 = np.array([[float(theta0_val)]])
    D = design_matrix.shape[1]
    X0_M = np.random.normal(loc=theta0_val, scale=1.0, size=(D, M))

    th_list, x_values = soul_mh_decay(
        log_p=log_p_laplace,
        th0=th0,
        x0_M=X0_M,
        y_l=design_matrix,
        y_f=labels,
        T=T,
        M=M,
        B=B,
        D=D,
        delta_step=delta_step,
        proposal_std=proposal_std,
        b=b_scale,
        gamma = 1
    )
    th_trajectory = np.array([t[0, 0] for t in th_list])
    return run_idx, theta0_val, th_trajectory, x_values


# Run all 7 initializations in parallel across available CPU cores (leave at least one free so the computer stays responsive)
results = Parallel(n_jobs=-2)(
    delayed(run_single_soul_mh_decay)(i) for i in range(7)
)

# Plot all 7 runs on one figure
fig = plt.figure(figsize=(10, 6))

theta_soul = []
X_soul = []

# Loop over the parallel results and add each line to the same plot
for run_idx, theta0_val, th_trajectory, x_values in results:
    theta_soul.append(th_trajectory)
    X_soul.append(x_values)

    plt.plot(
        th_trajectory,
        label=f"Run {run_idx + 1} (Init $\\theta_0={theta0_val}$)",
    )

# Reference line for ground truth
plt.axhline(
    y=np.mean(x_unknown),
    color="black",
    linestyle="dashed",
    label="True Mean $\\theta$",
)

plt.title("SOUL-MH Convergence Across Multiple Initializations ()") #$\\gamma = 0.9995 $
plt.xlabel("Iteration (T)")
plt.ylabel("Theta Estimate ($\\theta$)")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)

plt.show()
fig.savefig("soul_mh_convergence_parallel.pdf", format="pdf")

The relative error:

In [ ]:
# Convert list of trajectories into a 2D numpy array of shape (num_runs, T + 1)
theta_soul_arr = np.array(theta_soul)

# Compute relative error trajectory for each run
relative_error_trajectories = np.abs(theta_soul_arr - theta_true) / np.abs(
    theta_true
)

# Mean relative error across all runs per iteration step
mean_relative_error_per_step = np.mean(relative_error_trajectories, axis=0)

# Plot Relative Error over iterations
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
for idx, err_traj in enumerate(relative_error_trajectories):
    plt.plot(err_traj, alpha=0.4, label=f"Run {idx + 1}")

plt.plot(
    mean_relative_error_per_step,
    color="black",
    linewidth=2,
    label="Mean Relative Error",
)
plt.yscale("log")  # Log scale highlights convergence speed clearly
plt.title("SOUL-MH Relative Error Convergence")
plt.xlabel("Iteration (T)")
plt.ylabel("Relative Error (Log Scale)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

- Steady Initial Descent: The mean relative error (black line) decreases consistently across the first ~800 iterations, dropping from above $1.0$ down to around $0.2$ ($20\%$ error)
- Error Plateau: Around $T = 800$, the mean error hits a plateau at roughly $0.2$ ($20\%$). It stops improving systematically and begins oscillating.
- Run 1 started far out in a region where the gradients are very small (or where the random walk proposal rejected too frequently initially), causing it to lag behind within $T = 1500$ steps.

## SOUL Metropolis-Hastings with Standardised Scaling

In [ ]:
# @title SOUL MHSS Function
def soul_mh_ss(
    log_p,
    th0,
    x0_M,
    y_l,
    y_f,
    T,
    M,
    B,
    delta_step,
    proposal_std,
    lower_bounds,
    upper_bounds,
):
    """SOUL where latent sampling is done via Metropolis-Hastings with Standardised Scaling (MH SS)."""
    th = np.copy(th0)
    D = x0_M.shape[0]

    bounds_range = upper_bounds - lower_bounds

    x_t = np.copy(x0_M[:, 0:1]).reshape(D, 1)
    x_values = np.array(x0_M)
    th_list = [np.copy(th)]

    for t in range(1, T + 1):
        for m in range(1, M + 1):
            # Transform current position to [0, 1] standardised space
            x_t_std = (x_t - lower_bounds) / bounds_range

            z = np.random.normal(0.0, 1.0, x_t.shape)
            x_prop_std = x_t_std + proposal_std * z

            # proposals must stay within [0, 1]
            if np.any(x_prop_std < 0.0) or np.any(x_prop_std > 1.0):
                # Out of bounds proposal: target density is effectively 0 -> reject
                x_values = np.append(x_values, np.copy(x_t), axis=1)
                continue

            x_prop = x_prop_std * bounds_range + lower_bounds

            # Metropolis Hastings step
            log_alpha = min(
                0, log_p(th, x_prop, y_l, y_f) - log_p(th, x_t, y_l, y_f)
            )
            if np.log(np.random.uniform(0.0, 1.0)) < log_alpha:
                x_t = x_prop

            # Store the current position
            x_values = np.append(x_values, np.copy(x_t), axis=1)

        burnin_x_samples = x_values[:, -(M - B) :]

        avg_grad_th = np.zeros_like(th)
        for idx in range(M - B):
            x_m_burnin = burnin_x_samples[:, idx : idx + 1]
            avg_grad_th += (x_m_burnin - th).sum(0) / 5

        # Update theta
        th = th + delta_step * (avg_grad_th / (M - B))
        th_list.append(np.copy(th))

    return th_list, x_values

In [ ]:
# single run test

# --- SOUL MHSS Setup ---
D = 50
T = 800  # Outer optimization steps
M = 25  # MH steps per outer loop
B = 5  # Burn-in steps
delta_step = 0.08
proposal_std = 0.05

# Define reasonable bounds for latent variables X
lower_bounds = np.full((D, 1), -10.0)
upper_bounds = np.full((D, 1), 5.0)

# Initializations
th0 = np.array([[-9.0]])
x0_M = np.zeros((D, 1))

# Run the adapted SOUL-MHSS algorithm
th_list, x_samples = soul_mh_ss(
    log_p=log_p_laplace,
    th0=th0,
    x0_M=x0_M,
    y_l=design_matrix,
    y_f=labels,
    T=T,
    M=M,
    B=B,
    delta_step=delta_step,
    proposal_std=proposal_std,
    lower_bounds=lower_bounds,
    upper_bounds=upper_bounds,
)

print("Final estimated theta:", th_list[-1][0, 0])
print("True theta (loc parameter): -4.0")

In [ ]:
from joblib import Parallel, delayed

# --- Experiment Parameters ---
T = 1500  # Outer steps
B = 5
M = 20

# Hyper-parameters for SOUL-MHSS
delta_step = 0.05
proposal_std = 0.08

# Define bounds for the standardized scaling
D = design_matrix.shape[1]
lower_bounds = np.full((D, 1), -20.0)
upper_bounds = np.full((D, 1), 10.0)


def run_single_soul_mh_ss(run_idx):
    theta0_val = np.random.randint(-15, 10)
    th0 = np.array([[float(theta0_val)]])

    # Draw X0 within bounds
    X0_M = np.random.uniform(
        low=lower_bounds[0, 0], high=upper_bounds[0, 0], size=(D, M)
    )

    th_list, x_values = soul_mh_ss(
        log_p=log_p_laplace,
        th0=th0,
        x0_M=X0_M,
        y_l=design_matrix,
        y_f=labels,
        T=T,
        M=M,
        B=B,
        delta_step=delta_step,
        proposal_std=proposal_std,
        lower_bounds=lower_bounds,
        upper_bounds=upper_bounds,
    )

    th_trajectory = np.array([t[0, 0] for t in th_list])
    return run_idx, theta0_val, th_trajectory, x_values


# Run parallel executions
results = Parallel(n_jobs=-2)(
    delayed(run_single_soul_mh_ss)(i) for i in range(7)
)

# Plotting convergence
fig = plt.figure(figsize=(10, 6))

theta_soul_ss = []
X_soul_ss = []

for run_idx, theta0_val, th_trajectory, x_values in results:
    theta_soul_ss.append(th_trajectory)
    X_soul_ss.append(x_values)

    plt.plot(
        th_trajectory,
        label=f"Run {run_idx + 1} (Init $\\theta_0={theta0_val}$)",
    )

plt.axhline(
    y=np.mean(x_unknown),
    color="black",
    linestyle="dashed",
    label="True Mean $\\theta$",
)

plt.title("SOUL-MHSS Convergence Across Multiple Initializations (Parallel)")
plt.xlabel("Iteration (T)")
plt.ylabel("Theta Estimate ($\\theta$)")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)

plt.show()
fig.savefig("soul_mh_ss_convergence_parallel.pdf", format="pdf")

Relative error:

In [ ]:
# Convert list of trajectories into a 2D numpy array of shape (num_runs, T + 1)
theta_soul_ss_arr = np.array(theta_soul_ss)

# Compute relative error trajectory for each run
relative_error_trajectories_ss = np.abs(theta_soul_ss_arr - theta_true) / np.abs(
    theta_true
)

# Mean relative error across all runs per iteration step
mean_relative_error_per_step_ss = np.mean(relative_error_trajectories_ss, axis=0)

# Plot Relative Error over iterations
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
for idx, err_traj in enumerate(relative_error_trajectories_ss):
    plt.plot(err_traj, alpha=0.4, label=f"Run {idx + 1}")

plt.plot(
    mean_relative_error_per_step_ss,
    color="black",
    linewidth=2,
    label="Mean Relative Error",
)
plt.yscale("log")  # Log scale highlights convergence speed clearly
plt.title("SOUL-MHSS Relative Error Convergence")
plt.xlabel("Iteration (T)")
plt.ylabel("Relative Error (Log Scale)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()